# Bonsai-2 27B (PTQ1_0) on Colab CUDA
Runs **Ternary-Bonsai-2-27B-PTQ1_0** (5.54 GB) fully on GPU with the **Prism fork** llama.cpp
(prebuilt Linux CUDA binaries, release `prism-b10687-5d80cff`). Purpose: fix the 3.3 tok/s Vulkan crawl measured locally.

**Expected speed** (measure, don't trust): T4 ~5-15 tok/s, L4 ~15-40, A100 40GB ~30-80.
Free tier = T4, enough to validate the CUDA path.

**Steps:** Runtime > Change runtime type > **GPU (T4)**, then run cells top to bottom.
Cell 6 prints a public URL for your local machine. Session dies on disconnect; re-run from cell 2.


In [ ]:
# Cell 1: check GPU, pick matching CUDA build
import subprocess, re
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out)
m = re.search(r"CUDA Version:\s*([0-9.]+)", out)
drv = float(m.group(1)) if m else 0.0
CANDIDATES = [("12.8", TAG + "-bin-linux-cuda-12.8-x64.tar.gz"),
              ("12.4", TAG + "-bin-linux-cuda-12.4-x64.tar.gz")]
order = [c for c in CANDIDATES if drv >= float(c[0])] or [CANDIDATES[-1]]
print("driver CUDA:", drv, "-> try order:", [c[0] for c in order])

In [ ]:
# Cell 2: download prebuilt Prism llama.cpp (Linux CUDA), verify it runs
import os, glob, shutil, subprocess
ROOT="/content/llama-prism"; os.makedirs(ROOT, exist_ok=True)
def try_build(cudaver, tarname):
    tgz=f"/content/{tarname}"
    r=subprocess.run(["curl","-L","-q","-o",tgz,BASE+"/"+tarname])
    if r.returncode!=0: print("download failed", tarname); return False
    subprocess.run(["tar","-xzf",tgz,"-C",ROOT]); os.remove(tgz)
    hits=glob.glob(ROOT+"/**/llama-server", recursive=True)
    if not hits: print("no llama-server in tarball"); return False
    src=os.path.dirname(hits[0])
    for f in os.listdir(src):
        if os.path.isfile(os.path.join(src,f)): shutil.copy2(os.path.join(src,f),ROOT)
    env=dict(os.environ, LD_LIBRARY_PATH=ROOT+":"+os.environ.get("LD_LIBRARY_PATH",""))
    r=subprocess.run([ROOT+"/llama-server","--version"],capture_output=True,text=True,env=env)
    print((r.stdout.strip() or r.stderr.strip())[:300])
    return r.returncode==0
BIN=None
for cudaver,tarname in order:
    print(f"--- trying CUDA {cudaver} build ---")
    if try_build(cudaver,tarname): BIN=ROOT+"/llama-server"; print("OK ->",BIN); break
if BIN is None:
    print("all builds failed; installing CUDA runtime wheels and retrying")
    subprocess.run(["pip","install","-q","nvidia-cuda-runtime-cu12","nvidia-cublas-cu12"])
    import importlib.util
    wdirs=[]
    for wh in ["nvidia.cuda_runtime","nvidia.cublas"]:
        s=importlib.util.find_spec(wh)
        if s: wdirs.append(os.path.join(os.path.dirname(s.origin),"lib"))
    os.environ["LD_LIBRARY_PATH"]=ROOT+":"+":".join(wdirs)+":"+os.environ.get("LD_LIBRARY_PATH","")
    for cudaver,tarname in CANDIDATES:
        if try_build(cudaver,tarname): BIN=ROOT+"/llama-server"; break
assert BIN, "no usable build - file an issue with the output above"

In [ ]:
# Cell 3: download model (~5.54 GB, few minutes on Colab's pipe)
# Skip re-downloads across sessions: uncomment the Drive-cache block instead.
# from google.colab import drive; drive.mount('/content/drive')
# MODEL="/content/drive/MyDrive/models/Ternary-Bonsai-2-27B-PTQ1_0.gguf"
from huggingface_hub import hf_hub_download
MODEL = hf_hub_download("prism-ml/Ternary-Bonsai-2-27B-gguf",
                        "Ternary-Bonsai-2-27B-PTQ1_0.gguf", local_dir="/content/models")
print(MODEL)

In [ ]:
# Cell 4: launch server (full offload, 16K ctx, flash-attn). T4 16GB fits ~16K f16 KV.
# On A100 you can raise -c to 65536 or higher.
import subprocess, time, requests, os
API_KEY=""  # optional: set a string to require Authorization: Bearer <key> on all calls
args=[BIN,"-m",MODEL,"-ngl","99","-c","16384","-fa","on","--host","127.0.0.1","--port","8080"]
if API_KEY: args+=["--api-key",API_KEY]
log=open("/content/server.log","w")
P=subprocess.Popen(args,stdout=log,stderr=log)
hdr={"Authorization":"Bearer "+API_KEY} if API_KEY else {}
for i in range(120):
    try:
        r=requests.get("http://127.0.0.1:8080/health",headers=hdr,timeout=2)
        if r.ok: print("server ready after ~%ds"%(i*5)); break
    except Exception: pass
    time.sleep(5)
else:
    print("NOT READY - log tail:")
print(open("/content/server.log").read()[-800:])

In [ ]:
# Cell 5: speed benchmark (the number we came for)
import requests, time
def bench(msg,max_tokens=160):
    t0=time.time()
    r=requests.post("http://127.0.0.1:8080/v1/chat/completions",headers=hdr,json={
        "messages":[{"role":"user","content":msg}],"max_tokens":max_tokens,"temperature":0.7,
        "chat_template_kwargs":{"enable_thinking":False}})
    dt=time.time()-t0; u=r.json()["usage"]
    print(f"prompt {u['prompt_tokens']} tk @ {u['prompt_tokens']/max(dt-1,0.01):.1f} tk/s | "
          f"decode {u['completion_tokens']} tk @ {u['completion_tokens']/dt:.1f} tok/s | {dt:.1f}s")
    print(r.json()["choices"][0]["message"]["content"][:400])
bench("Count from 1 to 20, digits only.")
bench("Write a Python function that parses money strings like '$1,234.56' into cents.")


In [ ]:
# Cell 6: expose to your local machine (public trycloudflare URL)
import subprocess, time, re
Q=subprocess.Popen(["cloudflared","tunnel","--url","http://127.0.0.1:8080","--no-autoupdate"],
                   stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=None; t0=time.time()
while time.time()-t0<120:
    line=Q.stdout.readline()
    if line:
        print(line.strip())
        m=re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",line)
        if m: url=m.group(0); break
print("\nLOCAL TEST: open", (url or "TUNNEL-URL")+"/health", "in a browser")
print("Agent wiring: base_url =", (url or "TUNNEL-URL")+"/v1", " api_key =", API_KEY or "(none)")

## Stop / restart
- To relaunch: `!pkill -f llama-server` then rerun cell 4.
- After a Colab disconnect: rerun cells 2-6 (~4-5 min total; cell 2 skips nothing, cell 3 re-downloads unless Drive-cached).
- Report the tok/s numbers back so we can compare against local 3.3 tok/s and the card's 28-130 claims.